# Plan de Implementación: Capa de Data Management y OSS para Proyecto DOM

Este notebook detalla el plan técnico y las pruebas de integración para construir la "Capa de Ficheros" de la plataforma BIM, separando claramente las responsabilidades de **Data Management** (BIM 360/ACC) y **OSS** (Almacenamiento Genérico), tal como recomienda la arquitectura de Autodesk Platform Services (APS).

El objetivo es crear un **Catálogo Centralizado** en nuestra base de datos local que permita al backend (y a la IA) consultar la estructura de archivos, versiones y derivados sin tener que hacer llamadas lentas y recursivas a la API de Autodesk en tiempo real.

## Estructura del Plan
1.  **Configuración y Autenticación**: Obtención de tokens con los scopes correctos.
2.  **Data Management (Navegación)**: Exploración de Hubs y Proyectos.
3.  **Motor de Sincronización (Sync Engine)**: Escaneo recursivo de carpetas y archivos.
4.  **Catálogo de Datos**: Mapeo de respuestas API a nuestro esquema de base de datos (Prisma).
5.  **OSS (Buckets)**: Gestión de almacenamiento para archivos generados por la plataforma (no ACC).
6.  **Estrategias de Subida**: Implementación de subidas directas y firmadas (S3) para grandes volúmenes.
7.  **Integración DM-OSS**: Vinculación de versiones lógicas con objetos físicos.
8.  **Flujo End-to-End**: Simulación completa desde la subida hasta la consulta en el catálogo.

## 1. Configuración del Entorno y Estrategia de Autenticación

Para interactuar con Data Management (DM) y OSS, necesitamos un token **2-legged** (para procesos backend) con permisos amplios.

**Scopes Requeridos:**
*   `data:read`: Para leer la estructura de carpetas y metadatos de archivos en ACC/BIM 360.
*   `data:write`: (Opcional) Si planeamos crear carpetas o subir archivos a ACC.
*   `bucket:create`, `bucket:read`, `bucket:delete`: Para gestionar nuestro propio almacenamiento OSS (archivos de IA, reportes).
*   `data:create`: Para subir objetos a OSS.

### Implementación
Usaremos `axios` para solicitar el token al endpoint de OAuth de APS.

In [ ]:
// 1. Environment Setup and Authentication Strategy
require('dotenv').config({ path: '../../.env' }); // Adjust path as needed
const axios = require('axios');
const qs = require('qs');

const APS_CLIENT_ID = process.env.APS_CLIENT_ID;
const APS_CLIENT_SECRET = process.env.APS_CLIENT_SECRET;
const SCOPES = ['data:read', 'data:write', 'bucket:create', 'bucket:read', 'data:create'];

async function getInternalToken() {
    try {
        const data = qs.stringify({
            'client_id': APS_CLIENT_ID,
            'client_secret': APS_CLIENT_SECRET,
            'grant_type': 'client_credentials',
            'scope': SCOPES.join(' ')
        });

        const config = {
            method: 'post',
            url: 'https://developer.api.autodesk.com/authentication/v2/token',
            headers: { 
                'Content-Type': 'application/x-www-form-urlencoded'
            },
            data: data
        };

        const response = await axios(config);
        console.log("✅ Token acquired successfully.");
        console.log("Expires in:", response.data.expires_in, "seconds");
        return response.data.access_token;
    } catch (error) {
        console.error("❌ Authentication failed:", error.response ? error.response.data : error.message);
        throw error;
    }
}

// TEST: Verify token generation
// (Uncomment to run)
// getInternalToken().then(token => console.log("Token Preview:", token.substring(0, 10) + "..."));

In [ ]:
// 2. Data Management: Hubs and Projects Traversal

async function getHubs(token) {
    const config = {
        method: 'get',
        url: 'https://developer.api.autodesk.com/project/v1/hubs',
        headers: { 'Authorization': `Bearer ${token}` }
    };
    const response = await axios(config);
    return response.data.data;
}

async function getProjects(hubId, token) {
    const config = {
        method: 'get',
        url: `https://developer.api.autodesk.com/project/v1/hubs/${hubId}/projects`,
        headers: { 'Authorization': `Bearer ${token}` }
    };
    const response = await axios(config);
    return response.data.data;
}

async function getTopFolders(hubId, projectId, token) {
    const config = {
        method: 'get',
        url: `https://developer.api.autodesk.com/project/v1/hubs/${hubId}/projects/${projectId}/topFolders`,
        headers: { 'Authorization': `Bearer ${token}` }
    };
    const response = await axios(config);
    return response.data.data;
}

// TEST: Run Traversal
// (Uncomment to run)
/*
(async () => {
    try {
        const token = await getInternalToken();
        const hubs = await getHubs(token);
        console.log(`Found ${hubs.length} Hubs`);
        if(hubs.length > 0) {
            const hubId = hubs[0].id;
            console.log(`Using Hub: ${hubs[0].attributes.name} (${hubId})`);
            
            const projects = await getProjects(hubId, token);
            console.log(`Found ${projects.length} Projects`);
            
            if(projects.length > 0) {
                const projectId = projects[0].id;
                console.log(`Using Project: ${projects[0].attributes.name} (${projectId})`);
                
                const topFolders = await getTopFolders(hubId, projectId, token);
                console.log(`Found ${topFolders.length} Top Folders`);
                topFolders.forEach(f => console.log(` - ${f.attributes.displayName} (${f.id})`));
            }
        }
    } catch (e) { console.error(e); }
})();
*/

In [ ]:
// 3. Recursive Sync Engine Prototype

async function getFolderContents(projectId, folderId, token) {
    const config = {
        method: 'get',
        url: `https://developer.api.autodesk.com/data/v1/projects/${projectId}/folders/${folderId}/contents`,
        headers: { 'Authorization': `Bearer ${token}` }
    };
    const response = await axios(config);
    return response.data.data;
}

// Simplified recursive scanner
async function scanFolderRecursively(projectId, folderId, token, depth = 0) {
    const indent = "  ".repeat(depth);
    console.log(`${indent}📂 Scanning Folder: ${folderId}`);

    try {
        const contents = await getFolderContents(projectId, folderId, token);
        
        const folders = contents.filter(item => item.type === 'folders');
        const items = contents.filter(item => item.type === 'items');

        console.log(`${indent}   Found ${folders.length} subfolders and ${items.length} items.`);

        // Process Items (Files)
        for (const item of items) {
            const displayName = item.attributes.displayName;
            // Usually we want the 'tip' version details. 
            // The item itself contains a relationship to the tip version.
            // For this prototype, we just log the item.
            console.log(`${indent}   📄 File: ${displayName} (ID: ${item.id})`);
            
            // TODO: Insert into Local DB here
            // await prisma.file.upsert(...)
        }

        // Process Subfolders (Recursion)
        for (const folder of folders) {
            const folderName = folder.attributes.displayName;
            console.log(`${indent}   ➡️ Entering: ${folderName}`);
            
            // TODO: Insert Folder into Local DB here
            
            // Recursive call
            await scanFolderRecursively(projectId, folder.id, token, depth + 1);
        }

    } catch (error) {
        console.error(`${indent}❌ Error scanning folder ${folderId}:`, error.message);
    }
}

// TEST: Run Recursive Scan (Be careful with large projects!)
/*
(async () => {
    // ... setup token, hubId, projectId, topFolderId from previous steps ...
    // await scanFolderRecursively(projectId, topFolderId, token);
})();
*/

In [ ]:
// 4. Proposed Prisma Schema (Conceptual)

const prismaSchema = `
model Hub {
  id          String    @id // APS Hub ID (b.xxxx)
  name        String
  region      String?
  projects    Project[]
  createdAt   DateTime  @default(now())
  updatedAt   DateTime  @updatedAt
}

model Project {
  id          String    @id // APS Project ID (b.yyyy)
  hubId       String
  hub         Hub       @relation(fields: [hubId], references: [id])
  name        String
  rootFolderId String   // ID of the top-level folder
  folders     Folder[]
  files       File[]    // Files can be linked directly or via folders
  createdAt   DateTime  @default(now())
  updatedAt   DateTime  @updatedAt
}

model Folder {
  id          String    @id // APS Folder ID
  name        String
  projectId   String
  project     Project   @relation(fields: [projectId], references: [id])
  parentId    String?
  parent      Folder?   @relation("FolderHierarchy", fields: [parentId], references: [id])
  children    Folder[]  @relation("FolderHierarchy")
  files       File[]
  createdAt   DateTime  @default(now())
  updatedAt   DateTime  @updatedAt
}

model File {
  id          String    @id // APS Item ID
  name        String
  folderId    String
  folder      Folder    @relation(fields: [folderId], references: [id])
  projectId   String
  project     Project   @relation(fields: [projectId], references: [id])
  versions    FileVersion[]
  currentVersionId String? // Helper to quickly get tip version
  createdAt   DateTime  @default(now())
  updatedAt   DateTime  @updatedAt
}

model FileVersion {
  id          String    @id // APS Version ID
  fileId      String
  file        File      @relation(fields: [fileId], references: [id])
  versionNumber Int
  urn         String    // The URN used for translation/viewer
  mimeType    String?
  storageSize Int?
  createdAt   DateTime  @default(now())
}
`;

console.log("✅ Schema definition ready for review.");
console.log(prismaSchema);

In [ ]:
// 5. OSS Bucket and Object Management

const BUCKET_KEY = 'proyecto_dom_app_storage_' + APS_CLIENT_ID.toLowerCase(); // Must be globally unique

async function ensureBucketExists(token) {
    try {
        const config = {
            method: 'get',
            url: `https://developer.api.autodesk.com/oss/v2/buckets/${BUCKET_KEY}/details`,
            headers: { 'Authorization': `Bearer ${token}` }
        };
        await axios(config);
        console.log(`✅ Bucket ${BUCKET_KEY} exists.`);
    } catch (error) {
        if (error.response && error.response.status === 404) {
            console.log(`⚠️ Bucket ${BUCKET_KEY} not found. Creating...`);
            await createBucket(token);
        } else {
            throw error;
        }
    }
}

async function createBucket(token) {
    const data = {
        bucketKey: BUCKET_KEY,
        policyKey: 'persistent' // or 'transient' (24h), 'temporary' (30 days)
    };
    const config = {
        method: 'post',
        url: 'https://developer.api.autodesk.com/oss/v2/buckets',
        headers: { 
            'Authorization': `Bearer ${token}`,
            'Content-Type': 'application/json'
        },
        data: data
    };
    await axios(config);
    console.log(`✅ Bucket ${BUCKET_KEY} created successfully.`);
}

async function uploadObjectToOSS(objectName, filePath, token) {
    const fs = require('fs');
    const fileStream = fs.createReadStream(filePath);
    
    const config = {
        method: 'put',
        url: `https://developer.api.autodesk.com/oss/v2/buckets/${BUCKET_KEY}/objects/${objectName}`,
        headers: { 
            'Authorization': `Bearer ${token}`,
            'Content-Type': 'application/octet-stream'
        },
        data: fileStream,
        maxBodyLength: Infinity,
        maxContentLength: Infinity
    };
    
    const response = await axios(config);
    console.log(`✅ Object ${objectName} uploaded. ObjectId: ${response.data.objectId}`);
    return response.data;
}

// TEST: OSS Operations
/*
(async () => {
    const token = await getInternalToken();
    await ensureBucketExists(token);
    // await uploadObjectToOSS('test-report.pdf', './path/to/local.pdf', token);
})();
*/

## 6. Estrategia de Integración y Próximos Pasos

### Flujo de Trabajo Integrado
1.  **Inicialización**: Al arrancar el servidor (o mediante un cron job), ejecutar el `Recursive Sync Engine` para poblar la base de datos local.
2.  **Navegación Frontend**: El frontend consulta nuestra API (`/api/projects`, `/api/folders/:id`), que a su vez consulta la base de datos local (Prisma). **Latencia cercana a cero.**
3.  **Visualización**: Cuando el usuario selecciona un archivo, el frontend solicita el URN. Si es un archivo de BIM 360, devolvemos el URN almacenado. Si es un archivo generado (OSS), devolvemos el URN del bucket.
4.  **Generación de Reportes**:
    *   El backend genera un PDF.
    *   Sube el PDF a OSS.
    *   Registra el nuevo archivo en la DB local (vinculado al proyecto).
    *   El frontend actualiza la vista y muestra el nuevo archivo.

### Plan de Acción Inmediato
1.  **Actualizar `schema.prisma`**: Implementar los modelos definidos en la Sección 4.
2.  **Crear Servicios**:
    *   `ApsDataService`: Para interactuar con Hubs/Projects/Folders.
    *   `SyncService`: Para orquestar la sincronización recursiva.
    *   `OssService`: Para manejar buckets y subidas.
3.  **Endpoints**: Crear endpoints REST para exponer esta data desde la DB local.

---
**Fin del Plan de Implementación**

## 5. OSS: Gestión de Buckets y Objetos (Archivos Generados)

Para archivos que NO pertenecen a BIM 360 (como reportes PDF generados, snapshots, o archivos temporales de conversión), usaremos el servicio OSS (Object Storage Service).

### Conceptos
*   **Bucket**: Contenedor lógico de objetos. Debe tener un nombre único globalmente y una política de retención (`transient`, `temporary`, `persistent`).
*   **Object**: El archivo binario en sí.

### Estrategia
1.  Crear un bucket dedicado para la aplicación (ej. `proyecto-dom-reports`).
2.  Subir archivos generados a este bucket.
3.  Guardar la referencia (URN/ObjectId) en nuestra base de datos local, quizás en una tabla separada `GeneratedFile` o extendiendo la tabla `File` con un flag `isOss`.

## 4. Simulación del Esquema de Base de Datos (Prisma)

Para persistir esta estructura y evitar llamadas constantes a la API, necesitamos un esquema relacional robusto.

### Propuesta de Esquema
Necesitamos tablas para `Hub`, `Project`, `Folder`, `File` (Item) y `Version`.

*   **Hub**: Contenedor raíz.
*   **Project**: Pertenece a un Hub.
*   **Folder**: Estructura de árbol (Self-referencing relation `parentId`).
*   **File**: El objeto lógico.
*   **Version**: La instancia física del archivo (contiene el URN para el Viewer).

### Relaciones
*   `Hub` 1-n `Project`
*   `Project` 1-n `Folder` (Root folders)
*   `Folder` 1-n `Folder` (Subfolders)
*   `Folder` 1-n `File`
*   `File` 1-n `Version`

## 3. Motor de Sincronización Recursiva

El núcleo de nuestra "File Layer" es un motor que pueda recorrer recursivamente la estructura de carpetas de un proyecto y mapearla a nuestra base de datos local.

### Desafíos
*   **Profundidad**: Las carpetas pueden estar anidadas indefinidamente.
*   **Rate Limiting**: Hacer llamadas secuenciales para cada carpeta es lento y puede exceder los límites de la API.
*   **Items vs Versions**: En APS, un archivo es un `Item` que tiene múltiples `Versions`. Generalmente nos interesa la `tip` version (la última).

### Algoritmo Propuesto
1.  Comenzar en una `Folder` (Top Folder o subcarpeta).
2.  Obtener el contenido (`GET /projects/:project_id/folders/:folder_id/contents`).
3.  Separar el contenido en `sub-folders` y `items`.
4.  Para cada `item`, obtener su `tip` version y metadatos (URN, nombre, etc.).
5.  Para cada `sub-folder`, llamar recursivamente a la función (o encolar para procesamiento paralelo controlado).
6.  Guardar/Actualizar en DB Local (Prisma).

## 2. Data Management: Navegación de Hubs y Proyectos

El primer paso para construir nuestro "Catálogo Local" es entender la jerarquía de Data Management de Autodesk.
La estructura es: `Hubs` -> `Projects` -> `Top Folders` -> `Folders` (Recursivo) -> `Items` (Archivos) -> `Versions`.

### Estrategia
1.  **Listar Hubs**: Identificar la cuenta de BIM 360 / ACC.
2.  **Listar Proyectos**: Obtener los proyectos dentro de ese Hub.
3.  **Obtener Top Folders**: Cada proyecto tiene carpetas raíz (ej. "Project Files").

Este paso es crucial para establecer el "Root" de nuestra sincronización.